# NVIDIA Nemotron 3.5 Super VL — customer service with Microsoft Agent Framework

This notebook creates a **new managed Foundry agent** named `nemotron-customer-service-agent-framework` for the synthetic Northline Audio return case. Microsoft Agent Framework defines the agent, supplies the local `read_order_document` function, and runs the saved agent. The existing Northline file-search vector store supplies the returns policy and Aurora H7 support bulletin. This notebook **does not upload or duplicate documents**.

| Capability | This notebook |
| --- | --- |
| Document reasoning | Agent Framework's `OpenAIChatCompletionClient` sends the order image to NVIDIA Nemotron 3.5 Super VL. |
| Function calling | The saved Foundry agent requests `read_order_document`; Agent Framework executes it locally. |
| Knowledge | Foundry file search uses the two already-indexed policy documents. |
| Identity and inspection | Microsoft Entra credentials authorize calls; the saved agent, conversation, and traces remain in Foundry. |

The [NVIDIA article](https://developer.nvidia.com/blog/nvidia-nemotron-3-5-super-vl-powers-specialized-agents-with-multimodal-reasoning/) describes the same division of work: a document agent reads layout, a research agent connects evidence, and a customer-service agent follows policy. All customer and company records here are fictional.

**Why one Azure SDK call appears:** Agent Framework's `to_prompt_agent` converts the local Framework agent into a Foundry definition. Publishing that definition as a named, versioned resource currently uses `AIProjectClient.agents.create_version`. The customer interactions run through `FoundryAgent`. [Official publish-and-connect sample](https://github.com/microsoft/agent-framework/blob/main/python/samples/02-agents/providers/foundry/foundry_prompt_agents.py).

## Setup

Follow [the sample README](README.md) to set up this notebook's Python environment and Microsoft Entra authentication. Run the direct Foundry SDK notebook once to index the two synthetic knowledge documents, then export its vector store ID. Run cells in order. The notebook does not delete its agent or conversations.


In [ ]:
import os
from datetime import date, timedelta
from pathlib import Path

from IPython.display import Image, Markdown, display
from agent_framework import Agent, Content, Message, tool
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from agent_framework.openai import OpenAIChatCompletionClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import DefaultAzureCredential, get_bearer_token_provider

In [ ]:
PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").rstrip("/")
MODEL = os.environ.get("FOUNDRY_MODEL", "")
if not PROJECT_ENDPOINT or not MODEL:
    raise EnvironmentError("Set FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_MODEL before running this notebook.")
INFERENCE_BASE_URL = os.getenv(
    "FOUNDRY_OPENAI_BASE_URL",
    PROJECT_ENDPOINT.rstrip("/") + "/openai/v1/",
)
AGENT_NAME = os.getenv("FOUNDRY_AGENT_FRAMEWORK_AGENT_NAME", "nemotron-customer-service-agent-framework")
VECTOR_STORE_ID = os.environ.get("FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID", "")
if not VECTOR_STORE_ID:
    raise EnvironmentError("Set FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID from the direct SDK notebook output.")
ASSET_DIR = next(
    (p for p in [
        Path.cwd() / "customer_service_assets",
        Path.cwd() / "inference" / "msfoundry" / "nemotron-35-vl-super" / "customer_service_assets",
    ] if (p / "northline_order_return_case.png").is_file()),
    None,
)
if ASSET_DIR is None:
    raise FileNotFoundError("Open this notebook from its directory or the repository root.")
IMAGE_PATH = ASSET_DIR / "northline_order_return_case.png"
for filename in ("northline_returns_policy.txt", "aurora_h7_support_bulletin.txt"):
    if not (ASSET_DIR / filename).is_file():
        raise FileNotFoundError(ASSET_DIR / filename)
if not INFERENCE_BASE_URL.rstrip("/").endswith("/openai/v1"):
    raise ValueError("FOUNDRY_OPENAI_BASE_URL must end in /openai/v1/")
credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
await credential.get_token("https://ai.azure.com/.default")
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
print("Microsoft Entra identity ready")
print(f"New agent: {AGENT_NAME}")
print(f"Existing vector store: {VECTOR_STORE_ID}")

## 1. Customer document

The image combines an order table, a shipment timeline, and the customer's return request. The reported right-earcup fault and attempted reset are **unverified customer statements**.

In [ ]:
display(Image(filename=str(IMAGE_PATH), width=950))

## 2. Define the image-reading function

The Framework's Chat Completions client sends the PNG to the Nemotron deployment using an Entra token. This local function returns observed visual facts only. Foundry's managed agent will request the function by name; Agent Framework will run it in this notebook process.

In [ ]:
vision_client = OpenAIChatCompletionClient(
    model=MODEL,
    base_url=INFERENCE_BASE_URL,
    api_key=token_provider,
)

FUNCTION_PARAMETERS = {
    "type": "object",
    "properties": {},
    "required": [],
    "additionalProperties": False,
}

@tool(
    name="read_order_document",
    schema=FUNCTION_PARAMETERS,
)
async def read_order_document_async() -> str:
    """Read the customer-supplied order and return-case image; report only visible facts."""
    prompt = (
        "Read the attached order and return-case document. Transcribe the order ID, "
        "product, SKU, price, order date, delivery date from the shipment table, "
        "case review date, return-request date, reported issue, reported troubleshooting, "
        "and listed box contents. Distinguish recorded order/shipment facts from "
        "the customer's unverified statements. Do not apply any return policy. "
        "Mark any unreadable field as unknown."
    )
    image_message = Message(role="user", contents=[
        Content.from_text(prompt),
        Content.from_data(IMAGE_PATH.read_bytes(), "image/png"),
    ])
    response = await vision_client.get_response([image_message])
    facts = response.text
    if not facts or not facts.strip():
        raise RuntimeError("NVIDIA Nemotron 3.5 Super VL returned no document facts.")
    print("read_order_document — visual facts from Nemotron:", facts)
    return facts

visual_tool = read_order_document_async
if visual_tool.parameters() != FUNCTION_PARAMETERS:
    raise RuntimeError("The local read_order_document tool has an invalid parameter schema. Restart the kernel and run from the top.")

## 3. Create the new managed agent

Agent Framework describes the instructions, local function, and Foundry file search. `to_prompt_agent` converts them to a versioned Foundry definition; the Azure project client publishes that definition. The existing vector store is attached by ID. **No files are uploaded or indexed in this step.**

In [ ]:
INSTRUCTIONS = (
    "You are Northline Audio's customer-service returns specialist in a fictional case. "
    "For order, shipment, item, and customer-statement facts, call read_order_document "
    "unless those visual facts are already in the conversation. For policy, dates, "
    "shipping, fees, refund timing, troubleshooting, or escalation, use file search "
    "to consult both the Northline Audio Returns Policy and Aurora H7 No-Audio Support Bulletin. "
    "The case review date is 2026-10-21. Count calendar days from the day after delivery. "
    "Separate visual facts, unverified customer statements, policy rules, and support guidance. "
    "Name the source title in plain text after each nonvisual claim. "
    "Do not say a defect is confirmed, an RMA is issued, or a refund is approved. "
    "Only explain conditional eligibility and the steps a human support team must complete."
)
local_definition = Agent(
    client=FoundryChatClient(project_endpoint=PROJECT_ENDPOINT, model=MODEL, credential=credential),
    name=AGENT_NAME,
    description="Northline Audio returns specialist using Nemotron vision and Foundry policy search",
    instructions=INSTRUCTIONS,
    tools=[visual_tool, FoundryChatClient.get_file_search_tool(vector_store_ids=[VECTOR_STORE_ID])],
)
prompt_definition = to_prompt_agent(local_definition)
assert {item.type for item in prompt_definition.tools} == {"function", "file_search"}
function_tool = next(item for item in prompt_definition.tools if item.type == "function")
if function_tool.parameters != FUNCTION_PARAMETERS:
    raise RuntimeError("The Foundry agent definition lost the strict read_order_document schema. Restart the kernel and run from the top.")
async with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project:
    created = await project.agents.create_version(
        agent_name=AGENT_NAME,
        definition=prompt_definition,
        description=local_definition.description,
    )
    published = await project.agents.get_version(created.name, str(created.version))
published_functions = [item for item in published.definition.tools if item.type == "function"]
if len(published_functions) != 1 or published_functions[0].parameters != FUNCTION_PARAMETERS:
    raise RuntimeError("Foundry saved an invalid read_order_document schema. Do not run the question cells against this version.")
print(f"Published {created.name} version {created.version}")
print("Verified strict read_order_document schema in the saved Foundry agent")
print("Open https://ai.azure.com/ to inspect the new agent.")

## 4. Connect through Microsoft Agent Framework

`FoundryAgent` runs a saved version whose function schema has been verified in Foundry. If a previous notebook run left `created` pointing to an invalid version, this cell selects the newest compatible saved version and prints that choice. The function declaration and file-search configuration live in Foundry; passing `visual_tool` here supplies the local implementation for `read_order_document`. Agent Framework may log that it omits the local tool declaration from the request; that message is informational because Foundry already stores the declaration. Keep this notebook kernel running while demonstrating the function call.

In [ ]:
def compatible_agent_version(version):
    definition = version.definition
    functions = [item for item in definition.tools if item.type == "function"]
    searches = [item for item in definition.tools if item.type == "file_search"]
    return (
        definition.model == MODEL
        and definition.instructions == INSTRUCTIONS
        and len(functions) == 1
        and functions[0].name == "read_order_document"
        and functions[0].parameters.get("type") == "object"
        and functions[0].parameters.get("additionalProperties") is False
        and len(searches) == 1
        and VECTOR_STORE_ID in searches[0].vector_store_ids
    )

requested_version = str(created.version)
async with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project:
    selected_version = await project.agents.get_version(created.name, requested_version)
    if not compatible_agent_version(selected_version):
        selected_version = None
        async for candidate in project.agents.list_versions(created.name, limit=50):
            if compatible_agent_version(candidate):
                selected_version = candidate
                break
        if selected_version is None:
            raise RuntimeError("No saved agent version has a valid read_order_document schema and the expected file search. Restart the kernel and run from the top.")
        print(f"Version {requested_version} has an invalid saved tool schema; using verified version {selected_version.version}.")
active_agent_version = str(selected_version.version)
service_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=created.name,
    agent_version=active_agent_version,
    credential=credential,
    tools=[visual_tool],
    timeout=120.0,
)
session = service_agent.create_session()
print(f"Foundry agent version: {active_agent_version}")
print(f"Framework session: {session.session_id}")

## 5. Question runner

The three question cells below reuse one Framework session so the agent can carry the visual facts and retrieved policy forward. The conversation and tool output are retained for the live demo. A policy search can occasionally exhaust the model's output budget before producing text; the runner detects that specific incomplete response and asks the agent to finish using the retrieved evidence.

In [ ]:
async def get_saved_response(response_id: str):
    async with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project:
        response_client = project.get_openai_client()
        try:
            return await response_client.responses.retrieve(response_id)
        finally:
            await response_client.close()

async def ask_customer_agent(question: str):
    print("Question:", question)
    response = await service_agent.run(question, session=session, options={"max_tokens": 4096})
    if not response.text:
        saved = await get_saved_response(session.service_session_id)
        reason = getattr(saved.incomplete_details, "reason", None)
        if saved.status != "incomplete" or reason != "max_output_tokens":
            raise RuntimeError(f"The agent returned no final text (status={saved.status}, reason={reason}).")
        print("The response used its output budget after tool use; continuing with the retrieved evidence.")
        response = await service_agent.run(
            "Finish the previous answer using the visual facts and policy search results already in this conversation. "
            "Name both source titles, explain conditional eligibility, and do not promise an RMA or refund.",
            session=session,
            options={"max_tokens": 4096},
        )
    print("Framework session:", session.session_id)
    print("Foundry response ID:", session.service_session_id)
    print("Open https://ai.azure.com/ and search for this response ID in the agent Traces view.")
    if not response.text:
        saved = await get_saved_response(session.service_session_id)
        reason = getattr(saved.incomplete_details, "reason", None)
        raise RuntimeError(f"The agent returned no final text after continuation (status={saved.status}, reason={reason}).")
    display(Markdown(response.text))
    return response

## 6. Question 1 — interpret the order image

The new agent calls `read_order_document`. Nemotron identifies recorded order and shipment fields separately from the customer's unverified report.

In [ ]:
case_1 = await ask_customer_agent(
    "Read the attached Northline order and return-case document. What was ordered, "
    "when was it delivered, when was the return request submitted, and what does the "
    "customer say happened? Call read_order_document and separate recorded facts "
    "from unverified customer statements."
)

## 7. Question 2 — apply both knowledge sources

Python computes the day counts exactly; the agent uses Foundry file search for the policy and bulletin. In Foundry Traces, look for a completed `file_search` step and the source titles.

In [ ]:
delivery_date = date(2026, 9, 18)
review_date = date(2026, 10, 21)
standard_deadline = delivery_date + timedelta(days=30)
defect_review_deadline = delivery_date + timedelta(days=45)
print(
    f"Calendar check: {(review_date - delivery_date).days} days since delivery; "
    f"standard deadline {standard_deadline} ({(review_date - standard_deadline).days} days ago); "
    f"defect-review deadline {defect_review_deadline} "
    f"({(defect_review_deadline - review_date).days} days away)."
)
question_2_start_response_id = session.service_session_id
case_2 = await ask_customer_agent(
    f"As of {review_date}, is this case within the standard-return window or the "
    f"suspected-defect review window? Delivery was {delivery_date}; 33 calendar "
    f"days have elapsed. The standard deadline was {standard_deadline}, 3 days ago. "
    f"The defect-review deadline is {defect_review_deadline}, 12 days away. "
    "Search the Northline Audio Returns Policy and Aurora H7 No-Audio Support "
    "Bulletin in Foundry file search. Explain conditional eligibility, shipping, "
    "refund timing, and remaining troubleshooting. Name both source titles."
)
# A continuation can place file search in an earlier response from this question.
search_calls = []
response_id = session.service_session_id
for _ in range(10):
    if not response_id or response_id == question_2_start_response_id:
        break
    saved = await get_saved_response(response_id)
    search_calls.extend(item for item in saved.output if item.type == "file_search_call")
    response_id = saved.previous_response_id
if not search_calls:
    raise RuntimeError("The policy answer did not use Foundry file search.")
print(f"Verified {len(search_calls)} Foundry file-search call(s) for Question 2")


## 8. Question 3 — draft the customer reply

The answer should offer troubleshooting and a conditional support-review path without claiming that an RMA or refund has already been approved.

In [ ]:
case_3 = await ask_customer_agent(
    "Draft a concise customer-facing reply. Explain the remaining balance, firmware, "
    "and second-device checks, what support should do if the issue persists, and "
    "the conditional return path. Cite the returns policy and support bulletin by "
    "title. Do not say the RMA or refund is approved."
)

## Expected interpretation and demo boundaries

- The order image shows order **NA-28471**, Aurora H7 headphones (SKU **AH7-BLK**), delivered **2026-09-18**. The complaint and reset attempt are customer-reported.
- On **2026-10-21**, **33 calendar days** have elapsed. The ordinary 30-day window expired on **2026-10-18**; the 45-day suspected-defect review remains open through **2026-11-02**.
- The **Northline Audio Returns Policy** governs return windows, shipping, fees, and refund timing. The **Aurora H7 No-Audio Support Bulletin** governs technical checks and escalation.
- Support must still verify the issue and authorize any RMA. The agent must not promise a refund or prepaid label before the policy conditions are met.
- This notebook reuses an existing knowledge vector store, creates a distinct managed agent, and keeps its conversation for inspection. Traces require Application Insights and may appear after an ingestion delay.

[Framework Foundry Agent Service guide](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/agent-services/foundry) · [Framework Foundry model provider](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/model-providers/microsoft-foundry)